## Parsers
* String Output Parser
* Json Output Parser
* Structured Output Parser
* Pydantic output Parser


In [128]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from typing import Any, List, Annotated, Optional, Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

### String Output Parser

In [6]:
from langchain_core.output_parsers import StrOutputParser

In [7]:
# Model Preperation

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    temperature= 0.4
)

<B>Example Problem: </B>
First Genererate an answer for a human query, then send same to another LLM to get the feedback on the answer generated.

In [42]:
model = ChatHuggingFace(llm=llm)

In [41]:
queTemplate = PromptTemplate( template = '''You are an university student who is learning about Generative AI and has developed some basic level projects for his learning practice. Answer the following questions asked in your interview for GenAI role.
                                            Q1. {que}.
                                            ''',
                            input_variables=['que'],
                            validate_template= True)

In [44]:
interview = queTemplate.invoke({
    'que': 'Tell me about your experience in Gen AI field.'
})

In [45]:
response = model.invoke(interview)

In [47]:
response.content

"I'm excited to share my experience with you. As a university student, I've been actively learning about Generative AI for the past year, and I've developed several projects to practice and solidify my understanding of the concepts.\n\nI started by exploring the basics of Generative AI, including Generative Adversarial Networks (GANs), Variational Autoencoders (VAEs), and Recurrent Neural Networks (RNNs). I built a few simple projects to get a feel for these architectures, including a basic GAN that generated simple images and a VAE that compressed and decompressed images.\n\nAs I progressed, I delved deeper into the field and explored more advanced topics, such as style transfer, text-to-image synthesis, and music generation. I built a project that used a GAN to generate realistic images of faces, and another project that used a VAE to generate music based on a given style.\n\nOne of my most notable projects is a text-to-image synthesis model that uses a combination of GANs and VAEs t

In [48]:
feedbackTemplate = PromptTemplate( template = ''' You are a working professional in Genrative AI field, you are interviewing a fresher candidate. Judge thieir answers for given question.
                                  Answer: {answer}   
                                  Mark the answer correctness out of 10. 
                                  If Question or Answer are not know, reply "Question/Answer missing."                                  
                ''',
                input_variables=[ 'answer'])

In [49]:
question = 'Tell me about your experience in Gen AI field.'
answer = response.content

In [50]:
feedprompt = feedbackTemplate.invoke({'answer': answer})

In [51]:
response = model.invoke(feedprompt)

In [52]:
response

AIMessage(content="I'd rate the candidate's answer a 9 out of 10. Here's why:\n\n**Strengths:**\n\n1. **Demonstrated enthusiasm and passion**: The candidate shows genuine excitement about Generative AI, which is essential for a career in this field.\n2. **Clear explanation of learning journey**: The candidate provides a clear and concise explanation of their learning process, including the topics they've explored, the projects they've built, and the techniques they've used.\n3. **Showcasing projects and skills**: The candidate highlights several projects they've worked on, including text-to-image synthesis, style transfer, and music generation, which demonstrates their hands-on experience with Generative AI.\n4. **Staying up-to-date with the field**: The candidate mentions their efforts to stay current with the latest advancements in the field, including reading research papers, attending online courses, and participating in online communities.\n\n**Weaknesses:**\n\n1. **Lack of specif

In [31]:
strParser = StrOutputParser()

In [53]:
result = strParser.invoke(response)

In [54]:
result

"I'd rate the candidate's answer a 9 out of 10. Here's why:\n\n**Strengths:**\n\n1. **Demonstrated enthusiasm and passion**: The candidate shows genuine excitement about Generative AI, which is essential for a career in this field.\n2. **Clear explanation of learning journey**: The candidate provides a clear and concise explanation of their learning process, including the topics they've explored, the projects they've built, and the techniques they've used.\n3. **Showcasing projects and skills**: The candidate highlights several projects they've worked on, including text-to-image synthesis, style transfer, and music generation, which demonstrates their hands-on experience with Generative AI.\n4. **Staying up-to-date with the field**: The candidate mentions their efforts to stay current with the latest advancements in the field, including reading research papers, attending online courses, and participating in online communities.\n\n**Weaknesses:**\n\n1. **Lack of specific metrics or resu

### Below is a <B>Chain</B> elustration, will learn in future.

In [65]:
chain = queTemplate | model | strParser | feedbackTemplate | model | strParser 

In [66]:
result = chain.invoke({'que':question})

In [67]:
result

"I'll evaluate the candidate's answer based on its relevance, coherence, and depth of knowledge.\n\n**Correctness score: 8/10**\n\nHere's a breakdown of the strengths and weaknesses:\n\n**Strengths:**\n\n1. **Relevant experience**: The candidate has hands-on experience with popular libraries (TensorFlow, PyTorch, Stable Diffusion, and DALL-E), which is a great starting point.\n2. **Variety of projects**: The candidate has worked on multiple projects, including text-based chatbots, music generation, image generation, and face generation, demonstrating a broad understanding of Generative AI concepts.\n3. **Technical terms**: The candidate uses technical terms like Markov chains, GANs, RNNs, and VAEs, which shows they have a basic understanding of the underlying concepts.\n4. **Enthusiasm and motivation**: The candidate expresses excitement about learning and exploring the field, which is essential for a career in Generative AI.\n\n**Weaknesses:**\n\n1. **Lack of depth**: While the candid

## JSON Output Parser
* JSON parsers are light, universal and easy to paerse and read.
* JSON parser short defined schema design, its the LLM which sets up the schema for the parsed JSON output.


In [68]:
from langchain_core.output_parsers import JsonOutputParser

In [73]:
jsonParser = JsonOutputParser()

In [108]:
feedTemplate_json = PromptTemplate( template = ''' You are a working professional in Genrative AI field, you are interviewing a fresher candidate. Judge thieir answers for given question.
                                  Answer: {answer} 
                                  Focus on responding with ...
                                  Marks for correctness out of 10.
                                  Stenghts, weeknes, suggestions for improvement.
                                  {format_instruction}  
                                  "                                  
                ''',
                input_variables=[ 'answer'],
                partial_variables={"format_instruction": jsonParser.get_format_instructions()}) 

In [79]:
chain2 = queTemplate | model | strParser | feedTemplate_json | model | jsonParser

In [80]:
result_json = chain2.invoke(question)
result_json

{'answer_correctness': 8,
 'reasoning': {'strengths': ['The candidate has hands-on experience with various Generative AI techniques',
   'They have developed several projects to practice and solidify their understanding of the subject',
   'They demonstrate a good understanding of the fundamental concepts in Generative AI'],
  'weaknesses': ["The candidate's projects are still at a basic level, which may indicate a lack of depth in their understanding",
   "They don't provide specific details about their projects, such as the architectures used, the datasets employed, or the results achieved",
   'Their answer is more of a summary of their projects rather than a detailed explanation of their thought process and problem-solving skills']}}

## Structured Output Parser
* Helps extracting output from LLM responses bases on designed schema.
* Developer need to define a ResponseSchema, that model should return.

In [131]:
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain.chat_models import init_chat_model

In [116]:
schema = [ 
ResponseSchema(name= 'correctness', description= "Marks for the correctness of the answer", type= 'float'),
ResponseSchema(name= "feedback", description= "Over all Feedback for the answer", type= 'str'),
ResponseSchema(name='strength', description="Student's strenghts on the topic", type= 'str'),
ResponseSchema(name='weekness', description="Student's weekness on the topic", type= 'str' ),
ResponseSchema(name='suggestions', description="Suggest few areas for improvements", type= 'List[str]'),
ResponseSchema(name="result", description="Mark Passed or Failed, on his performace", type= "Literal['Pass', 'Fail']")
]

In [123]:
structuredParser = StructuredOutputParser.from_response_schemas(schema)

In [142]:
chain4 = queTemplate | model | strParser | feedTemplate_json | model | jsonParser

In [143]:
structured_output = chain4.invoke(question)

In [144]:
structured_output

{'marks_for_correctness': 8,
 'strengths': ['The candidate has a clear understanding of Generative AI concepts and techniques.',
  'They have hands-on experience with various projects, including text-to-image, music generation, and 3D model generation.',
  'The candidate is familiar with popular libraries and tools in the field, such as TensorFlow and PyTorch.'],
 'weaknesses': ["The candidate's response is a bit lengthy and meandering, which may make it difficult to follow.",
  'There is no clear focus on a specific area of Generative AI or a particular problem they are trying to solve.',
  'The candidate does not provide any concrete numbers or metrics to demonstrate the effectiveness of their projects.'],
 'suggestions_for_improvement': ['The candidate should focus on a specific area of Generative AI and explain how their experience and projects relate to that area.',
  'They should provide more concrete examples and metrics to demonstrate the effectiveness of their projects.',
  'T

* Important points to note:

* Working with Structured parsers make it neccessary to have same scema for parser and Output. Always use same schema fpr Output and parsers.
for example: 
* <B>OutputParserException:</B> Got invalid return object. Expected key <B>`correctness` to be present, but got {`marks_for_correctness`</B>: 8, 'strengths': ['The candidate has a good understanding of the basics of AI, machine learning, and deep learning.', 'They have hands-on experience with popular libraries like TensorFlow, PyTorch, and Keras.', 'The candidate has worked on several projects, including image generation, text-to-image synthesis, and 3D model generation.'], 'weaknesses': ["The candidate's answer is more of a summary of their experience rather than a direct response to the question.", "They don't explicitly mention how their experience in Gen AI field will benefit the company or the position they're applying for.", "The candidate's answer could be more concise and focused on the key points they want to convey."], 'suggestions_for_improvement': ['Ask more specific questions that require the candidate to think critically and provide concrete examples.', "Encourage the candidate to provide more details about their experience and how it relates to the company's goals and challenges.", 'Consider asking follow-up questions that allow the candidate to elaborate on their answers and provide more depth.']}

* In this case Json Schema cannot me set for output, hense the keys provided by the LLM does not matches and runtime error are raised by parser as <B>OutputParserException</B> 
* Difficult to enforce types in the StructuredOutputParser.


## PydanticeOutputParser

* Strict Schema Enforcement
* Type safe Allows Type coersion
* Easy validation


In [145]:
from langchain_core.output_parsers import PydanticOutputParser

In [195]:
class Feedback(BaseModel): 
    correctness : int = Field(gt=0, le=10, description= "Markes on correctness outof 10.")
    feedback: Annotated[str, "Over all Feedback for the answer"]
    strength: Annotated[str, "Student's strenghts on the topic"]
    weekness: Annotated[str, "Student's weekness on the topic"]
    suggestions:Annotated[List[str],"Suggest few areas for improvements"]
    result: Annotated[Literal["Pass", "Fail"], "Mark Passed or Failed, on his performace"]


In [161]:
chat_model = init_chat_model(model= "mistral-small-2603", configurable_fields = {
    'temperature' : 0.5, 
     })
modelFeedback = chat_model.with_structured_output(Feedback)

In [183]:
pydanticParser = PydanticOutputParser(pydantic_object=Feedback)

In [165]:
feedbackTemplate_pydantic  = PromptTemplate( template = ''' You are a working professional in Genrative AI field, you are interviewing a fresher candidate. Judge thieir answers for given question.
                                  Answer: {answer} 
                                  Focus on responding with ...
                                  Marks for correctness out of 10.
                                  Stenghts, weeknes, suggestions for improvement.
                                  {format_instruction}  
                                  "                                  
                ''',
                input_variables=[ 'answer'],
                partial_variables={"format_instruction": pydanticParser.get_format_instructions()}) 

In [185]:
chain3  = queTemplate | model | strParser | feedbackTemplate_pydantic | model | pydanticParser

In [167]:
question2 = "LLM applications are very prone to rubtime errors, explain some unexpected errors faced while working with Chat application. How did you overcome such scenarios."

In [186]:
pydanticResult = chain3.invoke({'que': question2}) 

In [196]:
for key, value in pydanticResult:
    print(f'{key} = {value}')


correctness = 8
feedback = The candidate demonstrated a good understanding of the concepts of out-of-vocabulary words, adversarial examples, and conversational flow management. However, there were some minor errors in the implementation and explanation.
strength = The candidate's ability to think critically and come up with creative solutions to the problems encountered is a significant strength.
weekness = The candidate's lack of experience with Generative AI and the limited scope of their projects are notable weaknesses.
suggestions = ['Consider working on more complex projects to gain hands-on experience with Generative AI.', 'Explore the use of more advanced techniques, such as transfer learning and multi-task learning, to improve the robustness of the model.', 'Practice explaining technical concepts in a clear and concise manner to improve communication skills.']
result = Pass
